# Tarea 3 (RL)
**Nombre:** Adrián Jesús Maldonado Oclica  

In [1]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA disponible: False


In [2]:
import gymnasium as gym

matches = [k for k in gym.registry.keys() if "MinAtar" in k]
print(matches[:20])

[]


In [3]:
import gymnasium as gym
from gymnasium.envs.registration import register

register(
    id="MinAtar/Breakout-v1",
    entry_point="minatar.envs.breakout:Env",
)

In [3]:
from src.common.envs import make_minatar_breakout, make_atari_breakout

env = make_minatar_breakout(seed=0)
obs, info = env.reset()
print("MinAtar obs shape:", obs.shape, "acciones:", env.action_space.n)
env.close()

env = make_atari_breakout(seed=0)
obs, info = env.reset()
print("Atari obs shape:", obs.shape, "acciones:", env.action_space.n)
env.close()

MinAtar obs shape: (4, 10, 10) acciones: 3
Atari obs shape: (4, 84, 84) acciones: 4


## Configuración general:

In [4]:
SEEDS = [0, 1, 2, 3, 4]

MINATAR_STEPS = 200_000
ATARI_STEPS = 500_000

NES_MINATAR_GENS = 200
GA_MINATAR_GENS = 200

NES_ATARI_GENS = 100
GA_ATARI_GENS = 100

## Test rapido:

In [8]:
!python -m src.ddqn.train_ddqn --env minatar --seed 0 --total_steps 10000 --eval_every 5000
!python -m src.ppo.train_ppo --env minatar --seed 0 --total_steps 10000 --eval_every 5000
!python -m src.evolutionary.train_nes_ga --method nes --env minatar --seed 0 --generations 5 --eval_every 1

[minatar] step=     92 | ep=  10 | retorno=   0.00 | avg20=   0.30 | eps=0.999 | loss=     nan
[minatar] step=    172 | ep=  20 | retorno=   1.00 | avg20=   0.25 | eps=0.998 | loss=     nan
[minatar] step=    262 | ep=  30 | retorno=   1.00 | avg20=   0.25 | eps=0.998 | loss=     nan
[minatar] step=    392 | ep=  40 | retorno=   1.00 | avg20=   0.50 | eps=0.996 | loss=     nan
[minatar] step=    482 | ep=  50 | retorno=   1.00 | avg20=   0.50 | eps=0.995 | loss=     nan
[minatar] step=    584 | ep=  60 | retorno=   1.00 | avg20=   0.35 | eps=0.994 | loss=     nan
[minatar] step=    674 | ep=  70 | retorno=   0.00 | avg20=   0.35 | eps=0.994 | loss=     nan
[minatar] step=    774 | ep=  80 | retorno=   2.00 | avg20=   0.35 | eps=0.993 | loss=     nan
[minatar] step=    854 | ep=  90 | retorno=   0.00 | avg20=   0.30 | eps=0.992 | loss=     nan
[minatar] step=    956 | ep= 100 | retorno=   0.00 | avg20=   0.30 | eps=0.991 | loss=     nan
[minatar] step=   1076 | ep= 110 | retorno=   0.00

------
# DDQN:

### DDQN propio en MinAtar

In [ ]:
for seed in SEEDS:
    !python -m src.ddqn.train_ddqn --env minatar --seed {seed} --total_steps {MINATAR_STEPS}

[minatar] step=     92 | ep=  10 | retorno=   0.00 | avg20=   0.30 | eps=0.999 | loss=     nan
[minatar] step=    172 | ep=  20 | retorno=   1.00 | avg20=   0.25 | eps=0.998 | loss=     nan
[minatar] step=    262 | ep=  30 | retorno=   1.00 | avg20=   0.25 | eps=0.998 | loss=     nan
[minatar] step=    392 | ep=  40 | retorno=   1.00 | avg20=   0.50 | eps=0.996 | loss=     nan
[minatar] step=    482 | ep=  50 | retorno=   1.00 | avg20=   0.50 | eps=0.995 | loss=     nan
[minatar] step=    584 | ep=  60 | retorno=   1.00 | avg20=   0.35 | eps=0.994 | loss=     nan
[minatar] step=    674 | ep=  70 | retorno=   0.00 | avg20=   0.35 | eps=0.994 | loss=     nan
[minatar] step=    774 | ep=  80 | retorno=   2.00 | avg20=   0.35 | eps=0.993 | loss=     nan
[minatar] step=    854 | ep=  90 | retorno=   0.00 | avg20=   0.30 | eps=0.992 | loss=     nan
[minatar] step=    956 | ep= 100 | retorno=   0.00 | avg20=   0.30 | eps=0.991 | loss=     nan
[minatar] step=   1076 | ep= 110 | retorno=   0.00

### DQN oficial SB3 en MinAtar:

In [ ]:
for seed in SEEDS:
    !python -m src.ddqn.train_sb3_dqn --env minatar --seed {seed} --total_steps {MINATAR_STEPS}

### Graficar DDQN vs SB3:

In [ ]:
!python -m src.ddqn.plot_ddqn_vs_sb3 \
  --ours_pattern "results/ddqn_minatar_seed*/eval_log.csv" \
  --sb3_pattern "results/sb3_dqn_minatar_seed*/evaluations.npz" \
  --title "DDQN propio vs DQN oficial SB3 en MinAtar Breakout" \
  --save_path "ddqn_vs_sb3_minatar.png"

In [ ]:
# Mostramos la figura:
img = plt.imread("ddqn_vs_sb3_minatar.png")
plt.figure(figsize=(9,5))
plt.imshow(img)
plt.axis("off")
plt.show()

### Obtenemos estadística DDQN vs SB3

In [ ]:
!python -m src.ddqn.pairwise_stats_ddqn_vs_sb3 \
  --ours_pattern "results/ddqn_minatar_seed*/eval_log.csv" \
  --sb3_pattern "results/sb3_dqn_minatar_seed*/evaluations.npz"

------------
# PPO
### PPO propio en MinAtar:

In [ ]:
for seed in SEEDS:
    !python -m src.ppo.train_ppo --env minatar --seed {seed} --total_steps {MINATAR_STEPS}

### PPO oficial SB3 en MinAtar:

In [ ]:
for seed in SEEDS:
    !python -m src.ppo.train_sb3_ppo --env minatar --seed {seed} --total_steps {MINATAR_STEPS}

### Graficar PPO vs SB3

In [ ]:
!python -m src.ppo.plot_ppo_vs_sb3 \
  --ours_pattern "results/ppo_minatar_seed*/eval_log.csv" \
  --sb3_pattern "results/sb3_ppo_minatar_seed*/evaluations.npz" \
  --title "PPO propio vs PPO oficial SB3 en MinAtar Breakout" \
  --save_path "ppo_vs_sb3_minatar.png"

In [ ]:
# Mostramos la figura:
img = plt.imread("ppo_vs_sb3_minatar.png")
plt.figure(figsize=(9,5))
plt.imshow(img)
plt.axis("off")
plt.show()

### Estadística PPO vs SB3:

In [ ]:
!python -m src.ppo.pairwise_stats_ppo_vs_sb3 \
  --ours_pattern "results/ppo_minatar_seed*/eval_log.csv" \
  --sb3_pattern "results/sb3_ppo_minatar_seed*/evaluations.npz"

---------------
# NES / GA
### NES en MinAtar:

In [ ]:
for seed in SEEDS:
    !python -m src.evolutionary.train_nes_ga --method nes --env minatar --seed {seed} --generations {NES_MINATAR_GENS}

### GA en MinAtar:

In [ ]:
for seed in SEEDS:
    !python -m src.evolutionary.train_nes_ga --method ga --env minatar --seed {seed} --generations {GA_MINATAR_GENS}

### Graficar NES:

In [ ]:
!python -m src.evolutionary.plot_nes_ga_learning_curves \
  --pattern "results/nes_minatar_seed*/eval_log.csv" \
  --title "NES en MinAtar Breakout" \
  --save_path "nes_minatar_curve.png"

### Graficar GA:

In [ ]:
!python -m src.evolutionary.plot_nes_ga_learning_curves \
  --pattern "results/ga_minatar_seed*/eval_log.csv" \
  --title "GA en MinAtar Breakout" \
  --save_path "ga_minatar_curve.png"

In [ ]:
# mostramos las figuras:
for path in ["nes_minatar_curve.png", "ga_minatar_curve.png"]:
    img = plt.imread(path)
    plt.figure(figsize=(9,5))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

------------
# Atari
### DDQN y PPO en Atari:

In [ ]:
for seed in SEEDS:
    !python -m src.ddqn.train_ddqn --env atari --seed {seed} --total_steps {ATARI_STEPS}
    !python -m src.ddqn.train_sb3_dqn --env atari --seed {seed} --total_steps {ATARI_STEPS}
    !python -m src.ppo.train_ppo --env atari --seed {seed} --total_steps {ATARI_STEPS}
    !python -m src.ppo.train_sb3_ppo --env atari --seed {seed} --total_steps {ATARI_STEPS}

### NES y GA en Atari:

In [ ]:
for seed in SEEDS:
    !python -m src.evolutionary.train_nes_ga --method nes --env atari --seed {seed} --generations {NES_ATARI_GENS}
    !python -m src.evolutionary.train_nes_ga --method ga --env atari --seed {seed} --generations {GA_ATARI_GENS}

----------
# Comparación de todos los modelos:


In [ ]:
!python -m src.analysis.compare_all_methods --results_dir results --output_dir analysis_outputs

### Tablas:

In [ ]:
desc = pd.read_csv("analysis_outputs/descriptive_stats.csv")
pairwise = pd.read_csv("analysis_outputs/pairwise_wilcoxon.csv")
ranks = pd.read_csv("analysis_outputs/average_ranks.csv")

display(desc)
display(pairwise)
display(ranks)

### Graficas globales:

In [ ]:
for path in [
    "analysis_outputs/combined_curves_minatar.png",
    "analysis_outputs/combined_curves_atari.png",
    "analysis_outputs/critical_difference.png",
]:
    img = plt.imread(path)
    plt.figure(figsize=(10,5))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

# Conclusiones

A partir de los experimentos realizados en MinAtar Breakout y Atari Breakout, se compararon métodos basados en valor, gradiente de política y búsqueda evolutiva.

En términos generales:
- DDQN permitió reducir el sesgo de sobreestimación respecto a DQN vanilla.
- PPO mostró un entrenamiento más estable y competitivo.
- NES y GA resultaron funcionales, aunque con menor eficiencia muestral en comparación con DDQN y PPO.
- Las comparaciones estadísticas mediante Wilcoxon rank-sum permitieron identificar diferencias significativas entre algunos pares de métodos.
- El ranking promedio y el diagrama de diferencias críticas ofrecieron una visión global del comportamiento relativo de todos los algoritmos evaluados.